# Evaluation on the simulated validation and test sets

Loads a trained B-net, predicts the depth profile of every validation and
test B-scan and evaluates the predictions with the functions in
`helper_functions/Metrics_experimental_data.py`:

- `calculate_errors`: pixel-wise MAE / MedAE / RMSE over all defect pixels,
- `plot_signed_errors_by_depth`: signed error histograms per ground-truth depth,
- `calculate_metrics_per_simulation`: one depth estimate per defect (median of
  the defect interior), errors per simulation and their mean,
- `calculate_median_depth_errors`: used directly to test how the result
  depends on the number of eroded edge pixels.

The B-scans of one simulation are stored as consecutive files, so stacking
the predictions row by row rebuilds the depth map of each simulation
(`ROWS_PER_SIM` rows each). Run the notebook from the repository root.

In [ ]:
import numpy as np
import torch
from matplotlib import pyplot as plt
from torch.utils.data import DataLoader
from tqdm import tqdm

from data.data_operators import BScanDepthDataset, ComposeBScanTransforms
from networks.Unets import BnetSmallKernelSmarterRefine
from helper_functions.helper_functions import NoiseAdditionExperiment
from helper_functions.Metrics_experimental_data import (
    calculate_errors,
    calculate_median_depth_errors,
    plot_signed_errors_by_depth,
    calculate_metrics_per_simulation,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Specimen and dataset properties.
THICKNESS_MM = 3.5                    # thickness of the simulated CFRP plate
ROWS_PER_SIM = 512                    # B-scans (image rows) per simulation
VAL_DEPTHS = [0.25, 0.45, 0.65, 0.85]  # defect depths in the validation set
TEST_DEPTHS = [0.15, 0.35, 0.55, 0.75] # defect depths in the test set

# Trained model and its training logs (written by train_unet_bscan.py).
model_dir = "trained_models/Bnet_refined"
model_path = f"{model_dir}/best_model_clean.pth"
train_log_path = f"{model_dir}/train_log.pt"
val_log_path = f"{model_dir}/val_clean_log.pt"

# B-scans, depth targets and the normalisation constant of the training set.
val_bscans = "Bscan_thermography_dataset/validation_rb/validation_bscans"
val_masks = "Bscan_thermography_dataset/validation_rb/validation_masks"
test_bscans = "Bscan_thermography_dataset/testing_rb/testing_bscans"
test_masks = "Bscan_thermography_dataset/testing_rb/testing_masks"
normalization_path = "Bscan_thermography_dataset/training_rb/normalization_params_heating_and_cooling.npz"

# Whole sequence (heating and cooling) as input, as during training.
cooling_phase = False

# Gaussian noise added to the simulated B-scans to imitate camera noise
# (standard deviation in temperature units).
noise_sigma = 0.065
noise_augmentation = ComposeBScanTransforms([NoiseAdditionExperiment(sigma=noise_sigma)])

## Model and training curves

In [ ]:
model = BnetSmallKernelSmarterRefine().to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

train_log = torch.load(train_log_path)
val_log = torch.load(val_log_path)

plt.semilogy(train_log, label="Train")
plt.semilogy(val_log, label="Validation")
plt.legend()
plt.grid(alpha=0.3)
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("Training and validation loss")
plt.show()

In [ ]:
def predict(bscan_dir, mask_dir):
    """Predicts the depth profile of every B-scan in a folder.

    Returns the predictions and the ground truth stacked row by row,
    both of shape [number of B-scans, 512].
    """
    dataset = BScanDepthDataset(
        bscan_dir=bscan_dir,
        depth_dir=mask_dir,
        transform=noise_augmentation,
        normalization_path=normalization_path,
        cooling_phase=cooling_phase,
    )
    loader = DataLoader(dataset, batch_size=16, shuffle=False)

    pred_all, mask_all = [], []
    with torch.no_grad():
        for X, mask in tqdm(loader):
            pred_all.append(model(X.to(device)).cpu())
            mask_all.append(mask)

    return torch.cat(pred_all, dim=0), torch.cat(mask_all, dim=0)


def show_simulation(pred_all, mask_all, n):
    """Predicted depth map, ground truth and absolute error of simulation n (from 1)."""
    rows = slice((n - 1) * ROWS_PER_SIM, n * ROWS_PER_SIM)
    panels = [
        (pred_all[rows], "Predicted depth"),
        (mask_all[rows], "Ground truth"),
        (torch.abs(pred_all[rows] - mask_all[rows]), "Absolute error"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (image, title) in zip(axes, panels):
        im = ax.imshow(image, vmin=0, vmax=1)
        ax.set_title(f"Simulation {n}: {title}")
        ax.set_xlabel("Pixel width")
        ax.set_ylabel("Pixel height")
    fig.colorbar(im, ax=axes, label="Normalised depth", shrink=0.8)
    plt.show()


def evaluate(pred_all, mask_all, depth_levels, n_show=5):
    """Runs all metrics and plots for one dataset split."""
    num_simulations = pred_all.shape[0] // ROWS_PER_SIM

    # Pixel-wise errors over all defect pixels, converted to mm.
    errors = calculate_errors(pred_all, mask_all)
    print(
        f"Pixel-wise | MAE: {errors['mae'] * THICKNESS_MM:.4f} mm | "
        f"MedAE: {errors['medae'] * THICKNESS_MM:.4f} mm | "
        f"RMSE: {errors['rmse'] * THICKNESS_MM:.4f} mm | "
        f"defect pixels: {errors['count']}\n"
    )

    # Signed error per ground-truth depth (systematic over/under-estimation).
    plot_signed_errors_by_depth(pred_all, mask_all, depth_levels=depth_levels, bins=700)

    # Predicted vs. true depth maps of the first simulations.
    for n in range(1, min(n_show, num_simulations) + 1):
        show_simulation(pred_all, mask_all, n)

    # One depth estimate per defect, errors per simulation.
    return calculate_metrics_per_simulation(
        pred_all,
        mask_all,
        rows_per_sim=ROWS_PER_SIM,
        num_simulations=num_simulations,
        thickness_mm=THICKNESS_MM,
    )


def erosion_sweep(pred_all, mask_all, erosion_values=range(0, 6)):
    """Defect-level MAE (mm) as a function of the number of eroded edge pixels.

    Shows how much of the error comes from the defect edges, where the
    thermal response is blurred by lateral heat diffusion.
    """
    num_simulations = pred_all.shape[0] // ROWS_PER_SIM
    mean_mae = []

    for erosion in erosion_values:
        mae_per_sim = []
        for i in range(num_simulations):
            rows = slice(i * ROWS_PER_SIM, (i + 1) * ROWS_PER_SIM)
            metrics, _, _ = calculate_median_depth_errors(
                pred_all[rows], mask_all[rows], erosion_pixels=erosion
            )
            mae_per_sim.append(metrics["mae"] * THICKNESS_MM)
        mean_mae.append(np.mean(mae_per_sim))
        print(f"Erosion {erosion:2d} px | mean MAE: {mean_mae[-1]:.4f} mm")

    plt.plot(list(erosion_values), mean_mae, "o-")
    plt.xlabel("Eroded edge pixels")
    plt.ylabel("Mean defect-level MAE [mm]")
    plt.grid(alpha=0.3)
    plt.show()

## Validation set

In [ ]:
pred_val, mask_val = predict(val_bscans, val_masks)
per_sim_val, mean_val = evaluate(pred_val, mask_val, VAL_DEPTHS)

In [ ]:
erosion_sweep(pred_val, mask_val)

## Test set

In [ ]:
pred_test, mask_test = predict(test_bscans, test_masks)
per_sim_test, mean_test = evaluate(pred_test, mask_test, TEST_DEPTHS)

In [ ]:
erosion_sweep(pred_test, mask_test)